# YouTube Mix to Playlist Converter

This notebook converts YouTube auto-generated mixes into real YouTube playlists.

## Prerequisites:
1. `client_secret.json` in the same folder (from Google Cloud Console)
2. A YouTube Mix URL (looks like: `https://www.youtube.com/watch?v=...&list=RD...`)

## Steps:
1. Run **Cell 2** to install dependencies (once)
2. Run **Cell 3** to import libraries
3. Run **Cell 4** to define the Song class and YouTubeExtractor
4. Run **Cell 5** to define the YouTubePlaylistCreator
5. Run **Cell 6** to extract songs from your mix
6. Run **Cell 7** to create the playlist and add songs



In [ ]:
# STEP 1: INSTALL DEPENDENCIES (Run this cell once)

# These libraries are needed for YouTube extraction and API access
!pip install yt-dlp google-api-python-client google-auth-httplib2 google-auth-oauthlib

print("Dependencies installed!")



In [ ]:
# STEP 2: IMPORT LIBRARIES

import os
import json
import time
from dataclasses import dataclass
from typing import List, Optional

# For extracting YouTube mix data (without downloading videos)
import yt_dlp

# For YouTube Data API v3 authentication and requests
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

print("All libraries imported!")



In [ ]:
# STEP 3: DEFINE DATA STRUCTURES and EXTRACTOR

@dataclass
class Song:
    # Represents a single song extracted from YouTube.
    # Attributes:
    #     title: Song title (e.g., "Bohemian Rhapsody")
    #     artist: Artist name (e.g., "Queen")
    #     youtube_id: YouTube video ID (e.g., "fJ9rUzIMcZQ")
    #     duration: Length in seconds
    title: str
    artist: str
    youtube_id: Optional[str] = None
    duration: Optional[int] = None

    def __str__(self):
        return f"{self.artist} - {self.title}"


class YouTubeExtractor:
    # Extracts song metadata from YouTube mixes using yt-dlp.
    # YouTube mixes are auto-generated playlists (URL contains andlist=RD...).
    # This class reads the mix without downloading any videos.

    def extract_mix(self, mix_url: str, max_songs: int = 50) -> List[Song]:
        # Extract all songs from a YouTube mix URL.
        # Args:
        #     mix_url: YouTube mix URL with andlist=RD... parameter
        #     max_songs: Maximum songs to extract (default 50, YouTube mixes usually have 25-50)
        # Returns:
        #     List of Song objects with artist, title, youtube_id, duration

        # yt-dlp options:
        #   quiet=True          -> Suppress console output
        #   extract_flat=True   -> Get metadata only, no download
        #   playlistend=50      -> Limit to first N songs
        #   ignoreerrors=True   -> Skip unavailable/private videos
        ydl_opts = {
            'quiet': True,
            'extract_flat': True,
            'playlistend': max_songs,
            'ignoreerrors': True,
        }

        songs = []

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            print(f"Extracting from: {mix_url[:60]}...")

            # This fetches the playlist metadata WITHOUT downloading anything
            info = ydl.extract_info(mix_url, download=False)

            # 'entries' contains each video in the mix
            if 'entries' not in info:
                print("No playlist entries found. Make sure the URL has andlist=RD...")
                return songs

            for entry in info['entries']:
                if not entry:
                    continue  # Skip deleted/private videos

                raw_title = entry.get('title', 'Unknown')
                video_id = entry.get('id')
                duration = entry.get('duration')

                # YouTube titles often follow "Artist - Title" format
                # We try to split them for cleaner playlist data
                artist, song_title = self._parse_title(raw_title)

                songs.append(Song(
                    title=song_title,
                    artist=artist,
                    youtube_id=video_id,
                    duration=duration
                ))

        print(f"Extracted {len(songs)} songs from the mix!")
        return songs

    def _parse_title(self, title: str) -> tuple:
        # Parse YouTube video titles into (artist, title) pairs.
        # Handles formats like:
        #   "Queen - Bohemian Rhapsody"
        #   "Queen Bohemian Rhapsody" (en-dash)
        #   "Queen Bohemian Rhapsody" (em-dash)
        #   "Queen | Bohemian Rhapsody"
        # Returns:
        #     (artist, title) tuple. If unparseable, returns ("Unknown", title)

        separators = [' - ', ' – ', ' — ', ' | ']

        for sep in separators:
            if sep in title:
                parts = title.split(sep, 1)
                if len(parts) == 2:
                    return parts[0].strip(), parts[1].strip()

        # Fallback: could not parse artist from title
        return "Unknown", title


# Test the extractor (we will use it in the next cell)
print("YouTubeExtractor class defined!")
print("   Usage: extractor = YouTubeExtractor()")
print("          songs = extractor.extract_mix('YOUR_MIX_URL')")



In [ ]:
# STEP 4: YOUTUBE PLAYLIST CREATOR (API)

class YouTubePlaylistCreator:
    # Creates YouTube playlists and adds songs using YouTube Data API v3.
    # Requires:
    #     - client_secret.json (OAuth 2.0 credentials from Google Cloud)
    #     - YouTube Data API v3 enabled in your Google Cloud project

    # OAuth scope: allows creating playlists and adding videos
    SCOPES = ['https://www.googleapis.com/auth/youtube']

    def __init__(self, client_secrets_file: str = "client_secret.json"):
        # Initialize and authenticate with YouTube API.
        # This will open a browser tab asking you to grant permission.
        # You will only need to do this once per session.
        # Args:
        #     client_secrets_file: Path to your downloaded client_secret.json
        self.youtube = self._authenticate(client_secrets_file)
        print("Successfully authenticated with YouTube API!")

    def _authenticate(self, secrets_file: str):
        # OAuth 2.0 authentication flow.
        # Steps:
        # 1. Reads client_secret.json
        # 2. Opens browser for user consent
        # 3. Receives auth token via local callback server
        # 4. Returns authenticated YouTube API service
        flow = InstalledAppFlow.from_client_secrets_file(
            secrets_file, 
            self.SCOPES
        )
        # port=0 lets the OS pick an available port
        credentials = flow.run_local_server(port=0)

        # 'youtube' is the service object for all API calls
        return build('youtube', 'v3', credentials=credentials)

    def create_playlist(self, title: str, description: str = "", privacy: str = "private") -> str:
        # Create a new YouTube playlist in your account.
        # Args:
        #     title: Playlist name (e.g., "My Awesome Mix")
        #     description: Optional description shown on YouTube
        #     privacy: One of 'private' (only you), 'unlisted' (link only), 'public' (everyone)
        # Returns:
        #     Playlist ID string (used to add songs later)
        request = self.youtube.playlists().insert(
            part="snippet,status",
            body={
                "snippet": {
                    "title": title,
                    "description": description
                },
                "status": {
                    "privacyStatus": privacy
                }
            }
        )
        response = request.execute()
        playlist_id = response['id']

        print(f"Created YouTube playlist: '{title}'")
        print(f"   Privacy: {privacy}")
        print(f"   ID: {playlist_id}")
        return playlist_id

    def add_songs(self, playlist_id: str, songs: List[Song]):
        # Search for each song on YouTube and add the best match to the playlist.
        # How it works:
        # 1. For each Song, searches YouTube for "artist title"
        # 2. Takes the first (most relevant) result
        # 3. Adds that video to your playlist
        # Args:
        #     playlist_id: The playlist ID from create_playlist()
        #     songs: List of Song objects to add

        print(f"Adding {len(songs)} songs to playlist...")

        for i, song in enumerate(songs, 1):
            # Search query combines artist and title
            search_query = f"{song.artist} {song.title}"

            search_response = self.youtube.search().list(
                q=search_query,
                part="id",
                maxResults=1,  # We only need the top result
                type="video"
            ).execute()

            # Check if search found anything
            if not search_response['items']:
                print(f"  [{i}/{len(songs)}] Not found on YouTube: {song}")
                continue

            # Get the video ID of the best match
            video_id = search_response['items'][0]['id']['videoId']

            # Add this video to the playlist
            self.youtube.playlistItems().insert(
                part="snippet",
                body={
                    "snippet": {
                        "playlistId": playlist_id,
                        "resourceId": {
                            "kind": "youtube#video",
                            "videoId": video_id
                        }
                    }
                }
            ).execute()

            print(f"  [{i}/{len(songs)}] Added: {song}")

            # Rate limiting: YouTube API has quota limits
            # 0.5s delay keeps us well under the limit
            time.sleep(0.5)

        print("All done! Check your YouTube playlists.")


print("YouTubePlaylistCreator class defined!")
print("   Usage: creator = YouTubePlaylistCreator('client_secret.json')")
print("          playlist_id = creator.create_playlist('My Mix')")
print("          creator.add_songs(playlist_id, songs)")



In [ ]:
# STEP 5: EXTRACT SONGS FROM YOUR YOUTUBE MIX

# PASTE YOUR YOUTUBE MIX URL HERE:
MIX_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID&list=RDYOUR_VIDEO_ID"

# How to get this URL:
# 1. Play any song on YouTube
# 2. Look in the right sidebar for "Mix" or "Playlist"
# 3. Click the mix to start playing it
# 4. Copy the URL from your browser address bar
#    It should contain "andlist=RD..." (RD = Radio/Mix)

# Create extractor instance
extractor = YouTubeExtractor()

# Extract up to 10 songs for testing (change to 50 for full mix)
songs = extractor.extract_mix(MIX_URL, max_songs=10)

# Display what we found
print("Extracted songs:")
for i, song in enumerate(songs, 1):
    print(f"   {i}. {song}")

# Save backup to JSON (optional, but recommended)
if songs:
    backup_data = [{
        "title": s.title,
        "artist": s.artist,
        "youtube_id": s.youtube_id,
        "duration_seconds": s.duration
    } for s in songs]

    with open("extracted_songs.json", "w", encoding="utf-8") as f:
        json.dump(backup_data, f, indent=2, ensure_ascii=False)

    print("Backup saved to: extracted_songs.json")



In [ ]:
# STEP 6: CREATE PLAYLIST AND ADD SONGS

# IMPORTANT: Make sure client_secret.json is in the same folder as this notebook!

# Initialize the API client (this opens a browser for login)
creator = YouTubePlaylistCreator("client_secret.json")

# Create a new playlist
playlist_name = "My Converted Mix"  # Change this to whatever you want!
playlist_id = creator.create_playlist(
    title=playlist_name,
    description="Auto-generated from YouTube mix",
    privacy="private"  # Options: "private", "unlisted", "public"
)

# Add all extracted songs to the playlist
creator.add_songs(playlist_id, songs)

print(f"Success! Your playlist '{playlist_name}' is ready on YouTube.")



## Troubleshooting

### "FileNotFoundError: client_secret.json"
- Make sure you downloaded the OAuth credentials from Google Cloud
- Place the file in the same folder as this notebook
- Rename it to `client_secret.json` if needed

### "No playlist entries found"
- Check that your URL contains `andlist=RD...` (the RD prefix means it is a mix)
- Regular playlists start with `andlist=PL...`, not `RD...`
- Try refreshing the YouTube page and copying the URL again

### Songs not found during adding
- YouTube search uses the parsed artist/title. If the original uploader used weird formatting, parsing might fail.
- Check `extracted_songs.json` to see what was extracted.
- You can manually edit the JSON and re-run the add_songs step.

### Rate limit / quota exceeded
- YouTube API has daily quotas. The script has built-in delays.
- If you hit limits, wait 24 hours or check your quota at: https://console.cloud.google.com/apis/api/youtube.googleapis.com/quotas

